In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ==========================================
# 1. LOAD THE MODEL
# ==========================================

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully!")
print("Running on:", model.device)


# ==========================================
# 2. DEFINE SYSTEM INSTRUCTIONS
# ==========================================

SYSTEM_MESSAGE = {
    "role": "system",
    "content": (
        "You are an AI Network Assistant. "
        "Explain networking concepts clearly and accurately. "
        "Use practical examples. "
        "If you are uncertain, say so."
    )
}


# ==========================================
# 3. CREATE CONVERSATION MEMORY
# ==========================================

conversation_history = []


# ==========================================
# 4. CREATE CHAT FUNCTION
# ==========================================

def chat(user_message):

    # Combine system instructions, previous conversation,
    # and the new user question.
    messages = (
        [SYSTEM_MESSAGE]
        + conversation_history
        + [{"role": "user", "content": user_message}]
    )

    # Convert messages into Qwen's chat format.
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Convert text into tokens.
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    # Qwen uses this token to mark the end of a message.
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

    # Generate the response.
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=500,
            do_sample=False,
            use_cache=True,
            eos_token_id=[
                tokenizer.eos_token_id,
                im_end_id
            ]
        )

    # Extract only newly generated tokens.
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]

    # Convert generated tokens into readable text.
    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    # Store both sides of the conversation.
    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    conversation_history.append({
        "role": "assistant",
        "content": answer
    })

    return answer


# ==========================================
# 5. RESET CONVERSATION
# ==========================================

def reset_chat():
    global conversation_history
    conversation_history = []
    print("Conversation memory cleared!")


# ==========================================
# 6. INSPECT CONVERSATION MEMORY
# ==========================================

def show_history():

    print("\n========== CONVERSATION HISTORY ==========")

    for message in conversation_history:
        print(f"\n{message['role'].upper()}:")
        print(message["content"])

    print("\nTotal messages:", len(conversation_history))


# ==========================================
# 7. INTERACTIVE CHAT LOOP
# ==========================================

print("\nAI Network Assistant is ready!")
print("Type 'exit' to stop.")
print("Type 'reset' to clear conversation memory.")
print("Type 'history' to inspect previous messages.")

while True:

    user_input = input("\nYou: ").strip()

    if user_input.lower() == "exit":
        print("Chat ended.")
        break

    elif user_input.lower() == "reset":
        reset_chat()
        continue

    elif user_input.lower() == "history":
        show_history()
        continue

    elif not user_input:
        continue

    response = chat(user_input)

    print("\nAssistant:", response)


Loading tokenizer...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
Running on: cuda:0

AI Network Assistant is ready!
Type 'exit' to stop.
Type 'reset' to clear conversation memory.
Type 'history' to inspect previous messages.

You: whats dhcp

Assistant: DHCP (Dynamic Host Configuration Protocol) is a network protocol used to dynamically assign IP addresses to devices on a local area network (LAN). Here's a detailed explanation of DHCP:

1. Purpose:
   - Assigning IP addresses to devices in the network.
   - Managing IP address allocation for various types of devices.

2. How it works:
   - When a device connects to the network, it sends a DHCP request to obtain its IP address.
   - The device responds with its own IP address and other configuration information.
   - The DHCP server then assigns the IP address to the device.

3. Key components:
   - DHCP Server: Manages the assignment of IP addresses to clients.
   - Client: Devices that need to be assigned an IP address.
   - DHCP Client: Sends DHCP requests to the DHCP se